# PDB Workstation · Re-docking con GNINA 1.3

Cuaderno complementario de la **PDB Drug Discovery Workstation** (CICATA-IPN).

Evalúa la calidad de una estructura del PDB como **sistema de docking** mediante el
re-docking de su ligando co-cristalizado, siguiendo la heurística de
**Domínguez-Ramírez et al. 2025** (*Quality over quantity*):

- Un **CNN score ≥ 0.9** indica un receptor de alta calidad para docking.
- El **RMSD** respecto a la pose cristalográfica es un dato de apoyo (no un filtro rígido).

Copia los valores de **CNN score** y **RMSD** que arroja este cuaderno a la pestaña
*Selección de Receptor* de la Workstation.

---
**Requisito:** activa la GPU en `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.


## 1. Comprobar GPU


In [ ]:
import subprocess, sys
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode != 0:
    print('⚠️  No hay GPU activa. Ve a Entorno de ejecución → Cambiar tipo de entorno → T4 GPU y reinicia.')
else:
    print(r.stdout.split('\n')[8] if len(r.stdout.split('\n'))>8 else r.stdout)
    print('✓ GPU lista')


## 2. Instalar GNINA 1.3 y dependencias

Descarga el binario precompilado (no requiere compilar, ~1 min) e instala Open Babel para
la preparación de archivos.


In [ ]:
import os, subprocess, urllib.request

# Open Babel para conversión/preparación de ligandos
print('Instalando Open Babel...')
subprocess.run(['apt-get','-qq','install','-y','openbabel'], capture_output=True)

# Binario precompilado de GNINA 1.3 (incluye casi todas las dependencias)
GNINA_URL = 'https://github.com/gnina/gnina/releases/download/v1.3/gnina'
if not os.path.exists('/usr/local/bin/gnina'):
    print('Descargando GNINA 1.3 (~300 MB)...')
    urllib.request.urlretrieve(GNINA_URL, '/usr/local/bin/gnina')
    os.chmod('/usr/local/bin/gnina', 0o755)

# Verificar
v = subprocess.run(['gnina','--version'], capture_output=True, text=True)
print(v.stdout[:200] if v.stdout else v.stderr[:200])
print('✓ GNINA instalado')


## 3. Funciones de preparación

Descarga la estructura desde RCSB, separa la proteína (solo ATOM) del ligando co-cristalizado
y filtra solventes/iones comunes.


In [ ]:
import urllib.request, os

SOLVENTS = {'HOH','WAT','DOD','SO4','PO4','EDO','MPD','GOL','PEG','DMS','ACT','CIT',
            'FMT','MES','EPE','BME','DTT','TLA','ACE','EOH','IMD','TRS','DIO','IPA',
            'NAG','MAN','GLC','BMA','FUC','GAL','ZN','MG','NA','CL','K','CA','MN'}

def fetch_pdb(pdb_id):
    pdb_id = pdb_id.strip().upper()
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    txt = urllib.request.urlopen(url).read().decode()
    with open(f'{pdb_id}.pdb','w') as f: f.write(txt)
    return txt

def list_ligands(txt):
    ligs = {}
    for l in txt.split('\n'):
        if l[:6].strip()=='HETATM':
            code = l[17:20].strip()
            if code not in SOLVENTS:
                ligs[code] = ligs.get(code,0)+1
    return ligs

def split_structure(txt, pdb_id, lig_code):
    '''Separa la proteína (solo ATOM) y UNA SOLA instancia del ligando.
       Muchas estructuras tienen varias copias del co-cristal (varios sitios o cadenas);
       re-dockear todas juntas rompe el cálculo. Tomamos la primera instancia completa,
       identificada por (cadena, número de residuo, altloc).'''
    prot = [l for l in txt.split('\n') if l[:6].strip() in ('ATOM','TER')]
    # Identificar todas las instancias del ligando: clave (chain, resSeq)
    instances = []
    for l in txt.split('\n'):
        if l[:6].strip()=='HETATM' and l[17:20].strip()==lig_code:
            chain=l[21:22]; resseq=l[22:26].strip()
            key=(chain,resseq)
            if key not in instances: instances.append(key)
    if not instances:
        return None, None, 0, 0
    # Tomar la primera instancia; preferir altloc en blanco o 'A'
    chain0, resseq0 = instances[0]
    lig=[]
    for l in txt.split('\n'):
        if l[:6].strip()=='HETATM' and l[17:20].strip()==lig_code:
            if l[21:22]==chain0 and l[22:26].strip()==resseq0:
                altloc=l[16:17]
                if altloc in (' ','A'):  # evitar conformaciones alternas duplicadas
                    lig.append(l)
    with open(f'{pdb_id}_protein.pdb','w') as f: f.write('\n'.join(prot)+'\nEND\n')
    with open(f'{pdb_id}_{lig_code}.pdb','w') as f: f.write('\n'.join(lig)+'\nEND\n')
    return f'{pdb_id}_protein.pdb', f'{pdb_id}_{lig_code}.pdb', len(lig), len(instances)

print('✓ Funciones cargadas')


## 4. Función de re-docking

Ejecuta GNINA usando `--autobox_ligand` (la caja se define a partir del propio co-cristal)
y parsea el **CNN score**, **CNN affinity**, **afinidad** y **RMSD** respecto a la pose original.


In [ ]:
import subprocess, re

def compute_rmsd(redock_sdf_gz, ref_lig_pdb):
    '''RMSD entre la mejor pose y el co-cristal usando obrms (Open Babel),
       que hace mapeo atómico simétrico correcto. Devuelve None si falla.'''
    try:
        # obrms: primer arg = referencia, segundo = poses a comparar
        r=subprocess.run(['obrms', ref_lig_pdb, redock_sdf_gz],
                         capture_output=True, text=True, timeout=120)
        # salida tipo: 'RMSD 0.834' o 'nombre  0.834' por pose; tomar el primer número
        for line in r.stdout.split('\n'):
            m=re.search(r'([0-9]+\.[0-9]+)', line)
            if m: return float(m.group(1))
    except Exception as e:
        print(f'    (obrms: {e})')
    return None

def redock(pdb_id, prot, lig, seed=0):
    out=f'{pdb_id}_redock.sdf.gz'
    cmd=['gnina','-r',prot,'-l',lig,'--autobox_ligand',lig,
         '--cnn_scoring','rescore','--seed',str(seed),'-o',out]
    res=subprocess.run(cmd, capture_output=True, text=True)
    log=res.stdout+res.stderr
    # Parsear la primera pose de la tabla de GNINA.
    # Orden real de columnas en GNINA 1.3: mode | affinity(kcal/mol) | CNNaffinity | CNNscore
    # (CNNscore es 0-1; CNNaffinity es -log[K], típicamente > 1)
    cnn=cnnaff=aff=None
    # Regex tolerante: captura todos los números (incl. notación científica y signos) de cada línea.
    NUM=r'[-+]?[0-9]*\.?[0-9]+(?:[eE][-+]?[0-9]+)?'
    in_table=False
    for line in log.split('\n'):
        # La tabla de resultados empieza tras una cabecera con 'affinity' y 'CNN'
        if re.search(r'affinity', line, re.I) and re.search(r'CNN', line, re.I):
            in_table=True; continue
        # La primera fila de datos comienza con el número de modo '1'
        m=re.match(r'\s*1\s+('+NUM+r')\s+('+NUM+r')\s+('+NUM+r')', line)
        if m:
            aff=float(m.group(1)); cnnaff=float(m.group(2)); cnn=float(m.group(3))
            if cnn>1.0 and cnnaff<=1.0:  # salvaguarda por rango
                cnn, cnnaff = cnnaff, cnn
            break
    # Fallback: si no se halló la tabla, buscar líneas etiquetadas (algunas versiones las imprimen)
    if cnn is None:
        for line in log.split('\n'):
            ms=re.search(r'CNNscore[:\s]+('+NUM+r')', line, re.I)
            ma=re.search(r'CNNaffinity[:\s]+('+NUM+r')', line, re.I)
            maf=re.search(r'[Aa]ffinity[:\s]+(-'+NUM+r')', line)
            if ms: cnn=float(ms.group(1))
            if ma: cnnaff=float(ma.group(1))
            if maf and aff is None: aff=float(maf.group(1))
    rmsd=compute_rmsd(out, lig)
    # Si no se pudo parsear el CNN score, mostrar las líneas candidatas del log para diagnóstico
    if cnn is None:
        print('    ⚠ No se parseó el CNN score. Líneas del log con números:')
        for line in log.split('\n'):
            if re.search(r'CNN', line, re.I) or re.match(r'\s*[0-9]+\s+-?[0-9]', line):
                print('      |', line.rstrip()[:90])
    return {'pdb':pdb_id,'lig':lig.split('_')[-1].replace('.pdb',''),
            'affinity':aff,'cnn_score':cnn,'cnn_affinity':cnnaff,'rmsd':rmsd,'log':log}

print('✓ Función de re-docking lista')


## 5A. Opción A — Pegar PDB IDs

Escribe los códigos PDB separados por coma (ej. los que seleccionaste en la Workstation).
El cuaderno descarga cada estructura, detecta su ligando principal y re-dockea.


In [ ]:
#@title Ingresar PDB IDs { display-mode: 'form' }
PDB_IDS = '4ZUD, 6H5W'  #@param {type:'string'}
LIGANDO_FORZADO = ''     #@param {type:'string'}
#@markdown Deja *LIGANDO_FORZADO* vacío para autodetectar el ligando más grande.

ids=[x.strip().upper() for x in PDB_IDS.split(',') if x.strip()]
resultados=[]
for pid in ids:
    print(f'\n=== {pid} ===')
    try:
        txt=fetch_pdb(pid)
    except Exception as e:
        print(f'  ✗ No se pudo descargar: {e}'); continue
    ligs=list_ligands(txt)
    if not ligs:
        print('  ✗ Sin ligandos no-solvente detectados'); continue
    lig_code = LIGANDO_FORZADO.strip().upper() if LIGANDO_FORZADO.strip() else max(ligs,key=ligs.get)
    prot,ligf,n,ninst=split_structure(txt,pid,lig_code)
    print(f'  Ligando: {lig_code} | {ninst} copia(s) en el cristal | usando 1 ({n} átomos) | otros: {list(ligs)}')
    print(f'  Re-dockeando...')
    r=redock(pid,prot,ligf)
    resultados.append(r)
    cnn=f"{r['cnn_score']:.3f}" if r['cnn_score'] is not None else 'n/d'
    rmsd=f"{r['rmsd']:.2f}" if r['rmsd'] is not None else 'n/d'
    print(f"  → CNN score: {cnn} | RMSD: {rmsd} Å")
print('\n✓ Listo')


## 5B. Opción B — Subir archivos

Si prefieres usar archivos propios (por ejemplo el ZIP descargado de la Workstation, o tus
propios `.pdb`), ejecútalo aquí. Sube un **PDB completo** por estructura; el cuaderno separa
proteína y ligando automáticamente.


In [ ]:
from google.colab import files
import os

print('Sube uno o más archivos .pdb (estructura completa con co-cristal):')
up=files.upload()
resultados_b=[]
for fname in up:
    if not fname.lower().endswith('.pdb'): 
        print(f'  (omitido {fname})'); continue
    pid=os.path.splitext(fname)[0].upper()
    txt=open(fname).read()
    ligs=list_ligands(txt)
    if not ligs:
        print(f'  ✗ {fname}: sin ligandos'); continue
    lig_code=max(ligs,key=ligs.get)
    print(f'\n=== {pid} === ligando {lig_code}')
    prot,ligf,n,ninst=split_structure(txt,pid,lig_code)
    print(f'  {ninst} copia(s) del ligando; usando 1 ({n} átomos)')
    r=redock(pid,prot,ligf)
    resultados_b.append(r)
    cnn=f"{r['cnn_score']:.3f}" if r['cnn_score'] is not None else 'n/d'
    rmsd=f"{r['rmsd']:.2f}" if r['rmsd'] is not None else 'n/d'
    print(f"  → CNN score: {cnn} | RMSD: {rmsd} Å")
print('\n✓ Listo')


## 6. Tabla resumen

Consolida los resultados de ambas opciones. **Copia CNN score y RMSD** a la pestaña
*Selección de Receptor* de la Workstation. Recuerda: CNN ≥ 0.9 = alta calidad.


In [ ]:
import pandas as pd
todos = (resultados if 'resultados' in dir() else []) + (resultados_b if 'resultados_b' in dir() else [])
if not todos:
    print('No hay resultados todavía. Ejecuta la opción A o B.')
else:
    df=pd.DataFrame([{
        'PDB':r['pdb'],'Ligando':r['lig'],
        'CNN score':round(r['cnn_score'],3) if r['cnn_score'] is not None else None,
        'CNN affinity':round(r['cnn_affinity'],3) if r['cnn_affinity'] is not None else None,
        'Afinidad (kcal/mol)':round(r['affinity'],2) if r['affinity'] is not None else None,
        'RMSD (Å)':round(r['rmsd'],2) if r['rmsd'] is not None else None,
        'Calidad':'Alta (≥0.9)' if (r['cnn_score'] or 0)>=0.9 else ('Media' if (r['cnn_score'] or 0)>=0.5 else 'Baja'),
    } for r in todos])
    df=df.sort_values('CNN score',ascending=False,na_position='last').reset_index(drop=True)
    display(df)
    df.to_csv('gnina_redocking_resultados.csv',index=False)
    print('\nGuardado como gnina_redocking_resultados.csv')


---
### Notas metodológicas

- El **CNN score** evalúa la calidad de la pose con la red neuronal de GNINA; es más confiable
  que la afinidad sola para juzgar si el receptor reproduce el modo de unión conocido.
- El **RMSD** se calcula con **obrms** (Open Babel), que hace el mapeo atómico simétrico
  correcto entre la pose dockeada y el co-cristal. Un re-docking exitoso da RMSD < 2 Å,
  aunque conviene interpretarlo junto al CNN score (el RMSD crudo puede engañar con grupos simétricos).
- **Importante:** el cuaderno re-dockea **una sola copia** del ligando. Muchas estructuras
  (como 6H5W, con múltiples sitios de omapatrilat) tienen varias copias del co-cristal;
  re-dockearlas todas juntas produce un CNN score artificialmente bajo.
- Las estructuras con **CNN score bajo** suelen tener el sitio ocluido o residuos faltantes,
  algo que la calidad cristalográfica global (R-free, RSCC) no siempre revela.

Cita: McNutt et al., *GNINA 1.3*, J. Cheminformatics 2025; Domínguez-Ramírez et al.,
*Quality over quantity*, Front. Bioinform. 2025.
